# Chapter 01 — Introduction to ML & Data Science Workflows
## Live Example: Exploring a Dataset

**Session 1 | Chapter 1 | Duration: ~15 min**

---

In this notebook we will:
1. Load the famous **Iris dataset**
2. Inspect its structure with pandas
3. Compute basic statistics
4. Visualize distributions and relationships
5. Build first intuitions — *before writing any ML algorithm*

> **Key message:** Understanding your data visually is step 1 of every ML project.

**Remember:** Traditional programming = Rules + Data → Output. Machine Learning = Data + Output → Rules.

## 0. Setup — Importing Libraries

These are the tools we'll use throughout the entire course.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris

# Make plots look nice
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (10, 6)

print('All libraries loaded successfully!')
print(f'pandas version: {pd.__version__}')
print(f'numpy version:  {np.__version__}')

## 1. Loading the Iris Dataset

The **Iris dataset** is the 'Hello World' of machine learning:
- Collected by botanist Edgar Anderson in 1936
- Made famous by statistician R.A. Fisher
- 150 flower samples from 3 iris species
- 4 measurements per flower (features)
- **Goal:** Can we predict the species from the measurements?

In [ ]:
# Load the dataset from sklearn (no internet needed!)
iris_raw = load_iris()

# Convert to a pandas DataFrame — our main data structure
df = pd.DataFrame(
    data=iris_raw.data,
    columns=iris_raw.feature_names
)

# Add the target column (species name instead of number — more readable)
df['species'] = [iris_raw.target_names[t] for t in iris_raw.target]

print('Dataset loaded!')
print(f'Shape: {df.shape}  →  {df.shape[0]} rows, {df.shape[1]} columns')

## 2. First Look — What Does the Data Look Like?

Always start by looking at a few rows. Never assume what data looks like.

In [ ]:
# Show the first 5 rows
df.head()

In [ ]:
# Data types and non-null counts — always check this!
df.info()

In [ ]:
# How many samples per class?
# In ML, we call this 'class distribution'
print('Samples per species:')
print(df['species'].value_counts())
print('\n→ Balanced dataset: 50 samples per class. That\'s ideal!')

## 3. Basic Statistics

The `.describe()` method gives you a statistical summary in one line.
Look at min/max/mean/std to understand the scale and spread of each feature.

In [ ]:
# Statistical summary of numerical columns
df.describe().round(2)

In [ ]:
# Are there any missing values?
print('Missing values per column:')
print(df.isnull().sum())
print('\n→ Great! No missing values in this dataset.')
print('   (Real-world data is rarely this clean — see Chapter 02!)')

## 4. Visualizing Distributions

**Why visualize?**  
Numbers lie (or at least hide things). Visualizations reveal the *shape* of data.

Let's look at the distribution of each feature, split by species.

In [ ]:
# Distribution plots for all four features, colored by species
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features = iris_raw.feature_names
colors = {'setosa': '#e74c3c', 'versicolor': '#3498db', 'virginica': '#2ecc71'}

for ax, feature in zip(axes.flat, features):
    for species, group in df.groupby('species'):
        ax.hist(group[feature], alpha=0.6, label=species,
                color=colors[species], bins=15, edgecolor='white')
    ax.set_title(f'Distribution of {feature}', fontsize=13)
    ax.set_xlabel(feature)
    ax.set_ylabel('Count')
    ax.legend()

plt.suptitle('Feature Distributions by Species', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\nObservation: Some features separate the species very well!')
print('→ Petal length and petal width look very promising.')

## 5. Visualizing Relationships Between Features

A **scatter plot** shows how two features relate to each other.
A **pair plot** (or scatter matrix) shows *all* pairwise relationships at once.

In [ ]:
# Pair plot — the single most useful exploration tool
pair_plot = sns.pairplot(
    df,
    hue='species',
    palette=colors,
    diag_kind='kde',     # kernel density on diagonal
    plot_kws={'alpha': 0.7, 's': 60}
)
pair_plot.fig.suptitle('Pair Plot — All Feature Relationships', y=1.02, fontsize=15)
plt.show()

In [ ]:
# Zoom in: the most informative scatter — petal length vs petal width
fig, ax = plt.subplots(figsize=(9, 7))

for species, group in df.groupby('species'):
    ax.scatter(
        group['petal length (cm)'],
        group['petal width (cm)'],
        label=species.capitalize(),
        color=colors[species],
        s=80, alpha=0.8, edgecolors='white', linewidths=0.5
    )

ax.set_xlabel('Petal Length (cm)', fontsize=13)
ax.set_ylabel('Petal Width (cm)', fontsize=13)
ax.set_title('Petal Length vs Petal Width by Species', fontsize=15)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print('\nWhat do you notice?')
print('→ Setosa is completely separated!')
print('→ Versicolor and Virginica overlap slightly.')
print('→ A machine learning model will learn this boundary automatically.')

## 6. Correlation — How Do Features Relate to Each Other?

Correlation tells us if two features move together.  
- **+1.0** = perfect positive correlation  
- **-1.0** = perfect negative correlation  
- **0.0** = no linear relationship  

In [ ]:
# Correlation heatmap (numerical features only)
numeric_df = df.drop(columns=['species'])
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,       # show the numbers
    fmt='.2f',        # 2 decimal places
    cmap='RdBu_r',    # red-blue diverging colormap
    center=0,
    vmin=-1, vmax=1,
    square=True,
    ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print('\nObservation:')
print('→ Petal length and petal width are highly correlated (0.96)')
print('→ High correlation can be useful: together they form a clear 2D cluster structure')
print('→ But in high-dimensional data, redundant features can hurt — more on this in Dimensionality Reduction!')

## 7. Sneak Preview: What ML Does With This

We've just done **Exploratory Data Analysis (EDA)** — without any algorithm.

Let's see what happens when we *actually* train a model — a preview of what's coming in Chapter 03.

We use **K-Nearest Neighbors (KNN)** — one of the simplest ML algorithms:
- Given a new flower, find the K closest flowers in the training data
- Let them "vote" on the species → majority wins

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
import numpy as np

# ── Prepare data ──────────────────────────────────────────────────────────
X = iris_raw.data[:, 2:4]  # Use only petal length & width (the 2 best features)
y = iris_raw.target

# Split into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── Train a simple model ───────────────────────────────────────────────────
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)           # ← this is where "learning" happens

# ── Evaluate ───────────────────────────────────────────────────────────────
accuracy = model.score(X_test, y_test)
print(f'Model Accuracy: {accuracy:.1%}')
print('→ The model correctly classified this fraction of unseen flowers.')

In [ ]:
# Visualize the decision boundary the model learned
h = 0.02  # mesh step size
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(10, 7))
cmap_light = plt.cm.RdYlGn
ax.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_light)

scatter_colors = ['#e74c3c', '#3498db', '#2ecc71']
for i, (name, color) in enumerate(zip(iris_raw.target_names, scatter_colors)):
    mask = y == i
    ax.scatter(X[mask, 0], X[mask, 1], label=name,
               color=color, edgecolors='black', linewidths=0.5, s=70, zorder=5)

ax.set_xlabel('Petal Length (cm)', fontsize=13)
ax.set_ylabel('Petal Width (cm)', fontsize=13)
ax.set_title(f'Decision Boundary Learned by KNN Model (Accuracy: {accuracy:.1%})', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print('\nThe shaded regions = what the model predicts for any new flower.')
print('The model learned this boundary from the training data.')
print('→ We will understand HOW this works in Chapter 03!')

## Summary

In this notebook we:
- Loaded and inspected a real dataset with pandas
- Computed basic statistics and checked for missing values
- Visualized distributions and feature relationships
- Got a sneak preview of what a trained ML model looks like

**Key insight:** The clusters we *saw visually* are the same boundaries the model *learned automatically*.

---

**Next: Chapter 02 — Data Selection, Cleaning & Preparing**  
Real data is never this clean. Let's learn how to deal with the mess.